# Lección 1.1 — Programación tradicional vs. aprendizaje a partir de ejemplos

> **Módulo 1 · ¿Qué es aprender para una máquina?** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 60–75 min (lección con ejercicios intercalados ≈ 40 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Módulo 0 (en especial la Lección 0.2). Si el código te resulta extraño, la Lección 0.4.

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar la diferencia entre **escribir las reglas** (programación tradicional) y **aprenderlas de ejemplos** (ML).
- Ver cómo un modelo descubre una regla (la fórmula de grados Celsius a Fahrenheit) solo a partir de datos.
- Decidir con una lista de comprobación **cuándo conviene usar ML y cuándo no**.
- Reconocer los límites del aprendizaje a partir de ejemplos: solo sabe lo que vio.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números o palabras.

## ▶️ Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_digits
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

plt.rcParams["figure.figsize"] = (7, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: ¿cómo le enseñas a un niño qué es un perro?

No le das una definición del tipo *"mamífero cuadrúpedo de la familia de los cánidos con hocico alargado…"*. Le **muestras perros**: "mira, un perro", "ese también", "no, ese es un gato". Después de unos cuantos ejemplos, el niño reconoce perros que **nunca había visto**: chihuahuas, gran danés, perros dibujados…

Hay dos maneras de lograr que una computadora haga algo:
- **Programación tradicional:** le escribes **las reglas** paso a paso (como una receta de cocina).
- **Machine Learning:** le das **ejemplos con la respuesta correcta** y ella descubre las reglas (como el niño con los perros).

### ✋ Ejercicio 1 — Escribe la receta de "perro"
Intenta escribir **3 reglas** que permitan distinguir un perro de un gato en una foto. Luego busca un contraejemplo para cada una.

✍️ *Mis reglas y contraejemplos:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algunas reglas típicas y sus problemas:
- *"Los perros son más grandes"* → un chihuahua es más pequeño que muchos gatos.
- *"Los gatos tienen orejas puntiagudas"* → un pastor alemán también.
- *"Los perros tienen el hocico largo"* → un bulldog o un pug, no.

Para cada regla aparece una excepción. Tú distingues perros de gatos sin esfuerzo, pero **no sabes explicar cómo lo haces**. Ese es justo el tipo de problema donde el ML brilla.
</details>

---
## 2. 📖 El concepto: se invierte la flecha

```
PROGRAMACIÓN TRADICIONAL
    datos  ─┐
            ├──►  [ programa ]  ──►  respuestas
    reglas ─┘         (lo escribe una persona)

MACHINE LEARNING
  1) Entrenamiento:
    datos      ─┐
                ├──►  [ algoritmo de aprendizaje ]  ──►  reglas  (= el MODELO)
    respuestas ─┘

  2) Uso (predicción):
    datos nuevos ──►  [ modelo ]  ──►  respuestas
```

En la programación tradicional, **tú pones las reglas**. En ML, **tú pones ejemplos con respuestas** y el algoritmo produce las reglas. A ese conjunto de reglas aprendidas lo llamamos **modelo**.

> 📝 Fíjate en que el ML **no elimina la programación**: alguien programó el algoritmo de aprendizaje. Lo que cambia es que **las reglas del problema concreto** (perros, spam, precios…) ya no las escribe nadie a mano.

### ✋ Ejercicio 2 — ¿Qué entra y qué sale?
Un banco quiere detectar transacciones fraudulentas. Completa:
- Con **programación tradicional**, el banco tendría que dar: ______ y ______.
- Con **ML**, el banco tendría que dar: ______ y ______. El resultado sería un ______.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Tradicional:** las transacciones (datos) y **las reglas** de fraude escritas por expertos ("si el monto es > X y el país es distinto al habitual…").
- **ML:** las transacciones **pasadas** (datos) y **si cada una fue fraude o no** (respuestas). El resultado sería un **modelo** que, ante una transacción nueva, estima si es fraude.
</details>

---
## 3. ▶️ Cómo funciona: una máquina descubre una fórmula

Vamos con un caso donde **conocemos** la regla, para poder comprobar si la máquina la descubre.

Para pasar de grados Celsius a Fahrenheit, la regla es: **F = C × 1.8 + 32**.

**La forma tradicional:** escribimos la regla.

In [ ]:
def celsius_a_fahrenheit(c):
    return c * 1.8 + 32          # la regla la escribimos nosotros

print(celsius_a_fahrenheit(100))

**La forma ML:** no le damos la regla. Solo le damos **6 ejemplos** de lecturas de dos termómetros (uno en °C y otro en °F) y le pedimos a un modelo de **regresión lineal** (Lección 5.1) que encuentre la relación.

In [ ]:
ejemplos = pd.DataFrame({
    "celsius":    [-10, 0, 10, 20, 30, 40],
    "fahrenheit": [14, 32, 50, 68, 86, 104],
})
ejemplos

### 🔮 Ejercicio 3 — ¿Encontrará el 1.8 y el 32?
Un modelo de regresión lineal busca una regla de la forma **F = C × (algo) + (otro algo)**. Solo con los 6 ejemplos de arriba:
1. ¿Qué números crees que encontrará?
2. ¿Qué predecirá para 100 °C (un valor que **no está** en los ejemplos)?

✍️ *Mis predicciones:*

In [ ]:
modelo = LinearRegression()
modelo.fit(ejemplos[["celsius"]], ejemplos["fahrenheit"])      # ← aquí APRENDE de los ejemplos

print(f"Regla aprendida: F = C × {modelo.coef_[0]:.2f} + {modelo.intercept_:.2f}")
print(f"Predicción para 100 °C: {modelo.predict(pd.DataFrame({'celsius': [100]}))[0]:.1f} °F")

<details><summary><b>👀 Ver respuesta</b></summary>

Encontró **F = C × 1.80 + 32.00** y predice **212.0 °F** para 100 °C, el punto de ebullición del agua. ¡Descubrió la fórmula exacta solo a partir de ejemplos!

Esos dos números (1.8 y 32) son los **parámetros** del modelo: lo que el modelo **aprende** de los datos. Volverás a ellos en la Lección 1.2.
</details>

### ✏️ Ejercicio 4 — Termómetros imperfectos
En la vida real, las mediciones tienen errores. La celda añade **ruido** a las lecturas en °F (como un termómetro barato que falla un poco). Cambia `RUIDO` a 0, 2 y 10 y observa la regla aprendida. 🔮 Antes de ejecutar: ¿qué crees que le pasará a la regla con más ruido?

In [ ]:
RUIDO = 2                # ✏️ prueba 0, 2 y 10 (desviación típica del error, en °F)
CUANTOS_EJEMPLOS = 6     # ✏️ luego, con RUIDO = 10, prueba 50 y 500 ejemplos

rng = np.random.default_rng(0)
c = np.linspace(-10, 40, CUANTOS_EJEMPLOS)
f_medido = c * 1.8 + 32 + rng.normal(0, RUIDO, CUANTOS_EJEMPLOS)

modelo_ruido = LinearRegression().fit(c.reshape(-1, 1), f_medido)
print(f"Regla aprendida: F = C × {modelo_ruido.coef_[0]:.2f} + {modelo_ruido.intercept_:.2f}   (la real: × 1.80 + 32.00)")

plt.scatter(c, f_medido, label="lecturas con ruido")
plt.plot(c, modelo_ruido.predict(c.reshape(-1, 1)), "r-", label="regla aprendida")
plt.xlabel("°C"); plt.ylabel("°F"); plt.legend(); plt.show()

✍️ *Lo que observé:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Con `RUIDO = 0` la regla es exacta (× 1.80 + 32.00).
- Con `RUIDO = 2` se aleja un poco: **× 1.80 + 32.24**.
- Con `RUIDO = 10` se aleja bastante: **× 1.78 + 33.18**.
- Con `RUIDO = 10` y **50 ejemplos**: **× 2.03 + 29.81**. ¡Peor! Con pocos datos ruidosos, el resultado depende mucho de la suerte.
- Con `RUIDO = 10` y **500 ejemplos**: **× 1.78 + 31.99**. El término fijo ya casi clava el 32 y la regla se estabiliza.

Dos lecciones enormes en un solo ejercicio: (1) los datos ruidosos hacen que el modelo aprenda una regla **aproximada**, no exacta; (2) **muchos más ejemplos compensan el ruido** (unos pocos más no siempre bastan). Es una idea que volverá una y otra vez (Módulo 2).
</details>

---
## 4. 📖 ¿Qué pasa si la regla es imposible de escribir… o trivial?

### Caso A: reglas imposibles de escribir
Para una computadora, una imagen es solo una tabla de números. Este es un "3" escrito a mano, tal como lo ve la máquina (0 = blanco, 16 = negro):

In [ ]:
digitos = load_digits()
print(digitos.images[3].astype(int))
plt.figure(figsize=(2, 2)); plt.imshow(digitos.images[3], cmap="gray_r"); plt.axis("off"); plt.show()

Escribir reglas del tipo "si el píxel (2,5) es > 10 y el píxel (4,3)…" para reconocer los 10 dígitos, con todas las formas de escribirlos, es inviable. En la Lección 0.1 viste que un modelo, a partir de ejemplos, acierta el 98 %. **Aquí el ML gana por goleada.**

### Caso B: reglas triviales
¿Y al revés? Le pedimos a un modelo que aprenda si un número es **par**, a partir de ejemplos del 0 al 99. Luego lo probamos con números del 100 al 199.

### 🔮 Ejercicio 5 — ¿Aprenderá algo tan fácil?
1. ¿Qué acierto esperas con los números 0–99 (los que vio)?
2. ¿Y con los números 100–199 (nuevos)?

✍️ *Mis predicciones:*

In [ ]:
vistos = np.arange(0, 100).reshape(-1, 1)
nuevos = np.arange(100, 200).reshape(-1, 1)
es_par = lambda numeros: (numeros.ravel() % 2 == 0).astype(int)

arbol = DecisionTreeClassifier(random_state=0).fit(vistos, es_par(vistos))
print(f"Acierto con 0–99 (vistos)   : {arbol.score(vistos, es_par(vistos)):.0%}")
print(f"Acierto con 100–199 (nuevos): {arbol.score(nuevos, es_par(nuevos)):.0%}")
print(f"Profundidad del árbol: {arbol.get_depth()} preguntas seguidas  🤯")

<details><summary><b>👀 Ver respuesta</b></summary>

**100 % con los vistos y 50 % con los nuevos** (lo mismo que tirar una moneda). El árbol **memorizó** los 100 números con 99 preguntas seguidas ("¿es ≤ 0.5?, ¿es ≤ 1.5?…") en lugar de descubrir la idea de "par". Con números que nunca vio, no sabe qué hacer.

La regla tradicional `numero % 2 == 0` es **una línea**, acierta el 100 % **siempre** y no necesita datos. Moraleja: **si conoces una regla simple y exacta, úsala**. El ML es para cuando no la conoces.

(Este modelo, además, cayó en dos trampas que estudiarás a fondo: **sobreajuste**, Lección 4.4, y **no extrapolar** fuera de lo que vio, Lección 5.4.)
</details>

---
## 5. 📖 ¿Cuándo usar ML? Una lista de comprobación

Antes de lanzarte a entrenar un modelo, pregúntate:

| # | Pregunta | Si la respuesta es "no"… |
|---|---|---|
| 1 | ¿Existe un **patrón** que aprender? | Si es puro azar (la lotería), ningún modelo ayuda |
| 2 | ¿Es **difícil escribir las reglas** a mano? | Si hay una regla simple y exacta, prográmala |
| 3 | ¿Hay **datos** suficientes (idealmente, con la respuesta correcta)? | Sin datos no hay aprendizaje |
| 4 | ¿Se puede **tolerar algún error**? | El ML siempre se equivoca algunas veces |
| 5 | ¿El patrón **cambia** con el tiempo o hay que adaptarse a cada persona? | Si no cambia nunca, unas reglas pueden bastar (aunque el ML también sirve) |

Las tres primeras son imprescindibles; la 4 casi siempre; la 5 es una ventaja extra del ML.

### ✋ Ejercicio 6 — ¿ML o reglas?
Para cada problema, decide si usarías **ML** o **reglas** (programación tradicional) y qué pregunta de la lista lo decide:
1. Calcular el IVA de una factura.
2. Detectar transacciones fraudulentas con tarjeta.
3. Decidir si un alumno aprobó (nota final ≥ 6).
4. Recomendar canciones a cada usuario.
5. Comprobar que un correo electrónico tiene una "@" y un dominio.
6. Estimar en cuántos minutos llegará un pedido a domicilio.
7. Predecir el número ganador de la lotería.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Reglas** (pregunta 2: la fórmula es exacta y conocida).
2. **ML** (patrón complejo, muchos datos, los estafadores cambian de táctica: preguntas 2, 3 y 5).
3. **Reglas** (pregunta 2: una comparación).
4. **ML** (cada persona es distinta y hay muchísimos datos de escuchas: preguntas 2, 3 y 5).
5. **Reglas** (pregunta 2: un patrón de texto bien definido).
6. **ML** (depende del tráfico, la hora, el clima, el restaurante…: preguntas 2 y 3; un error de unos minutos es tolerable: pregunta 4).
7. **Ninguno** (pregunta 1: es azar puro, no hay patrón).
</details>

---
## 6. 🚨 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Usar ML donde basta una regla | Más caro, menos exacto y más difícil de mantener | Aplica la lista de la Sección 5 |
| "El modelo aprenderá aunque los datos sean malos" | **Basura entra, basura sale** (*garbage in, garbage out*) | Revisa la calidad de los datos (Lección 2.2) |
| Esperar 100 % de acierto | Con datos reales siempre hay ruido y casos raros | Define antes qué error es aceptable (Lección 4.8) |
| Creer que el modelo "entiende" | Solo aprendió patrones de **los ejemplos que vio** | Comprueba con datos nuevos, distintos a los de entrenamiento |
| Olvidar que los ejemplos pueden tener sesgos | El modelo copia los prejuicios del pasado | Revisa de dónde vienen los datos (Lección 15.4) |

### 🔮 Ejercicio 7 — Solo sabe lo que vio
Entrenas un reconocedor de perros **solo con fotos de labradores tomadas de día en un parque**. ¿Qué crees que pasará cuando le muestres un chihuahua de noche en un sofá? ¿Qué harías para evitarlo?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Probablemente fallará. El modelo aprendió patrones de **sus** ejemplos: tal vez "perro = animal dorado grande con césped verde detrás". Un chihuahua oscuro sobre un sofá no se parece a nada de lo que vio.

Solución: que los ejemplos de entrenamiento **representen la variedad** del mundo donde se usará el modelo (razas, tamaños, fondos, iluminación). Es la versión ML del refrán "dime con qué datos entrenas y te diré qué aprendes".
</details>

---
## 7. 🧠 Resumen en 5 puntos
1. **Programación tradicional:** datos + reglas → respuestas. **ML:** datos + respuestas → reglas (el **modelo**).
2. Un modelo puede descubrir una regla exacta (°C → °F) o una aproximada si los datos tienen ruido; **más ejemplos** ayudan.
3. El ML brilla cuando las reglas son **imposibles de escribir** (imágenes, lenguaje, conducta humana).
4. Si existe una **regla simple y exacta**, úsala: el ML puede memorizar en lugar de entender.
5. Un modelo **solo sabe lo que vio**: la calidad y variedad de los ejemplos lo decide todo.

---
## 8. 🏋️ Ejercicios finales

### 8.1 Repaso rápido (opción múltiple)

**1.** En Machine Learning, ¿qué produce el entrenamiento?
- a) Los datos · b) Las respuestas · c) El modelo (las reglas aprendidas) · d) El algoritmo

**2.** El modelo de °C → °F aprendió los números 1.8 y 32. ¿Cómo se llaman?
- a) Hiperparámetros · b) Parámetros · c) Etiquetas · d) Características

**3.** ¿Para cuál de estas tareas es **menos** recomendable el ML?
- a) Traducir textos · b) Calcular el área de un rectángulo · c) Detectar spam · d) Reconocer voces

**4.** Un modelo entrenado con datos muy ruidosos y pocos ejemplos…
- a) Aprende la regla exacta · b) Aprende una regla aproximada · c) No puede entrenarse · d) Siempre acierta el 100 %

**5.** ¿Por qué el árbol del Ejercicio 5 falló con los números 100–199?
- a) Porque esos números eran demasiado grandes para Python · b) Porque memorizó los ejemplos en lugar de captar la idea de "par" · c) Porque le faltaba un parámetro · d) No falló

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** · 2. **b)** Parámetros (los aprende el modelo; Lección 1.2) · 3. **b)** Hay una fórmula exacta: base × altura · 4. **b)** · 5. **b)**
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** Un hospital tiene un sistema de reglas para priorizar pacientes en urgencias que funciona bien desde hace años. Un proveedor les ofrece "cambiarlo por IA". ¿Qué preguntas harías antes de aceptar?

**B.** ¿Por qué crees que los filtros de spam modernos usan ML y no una lista de palabras prohibidas? Piensa en lo que hacen los que envían spam cuando descubren la lista.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**A.** Por ejemplo: ¿qué problema concreto tiene el sistema actual (qué errores comete)? ¿Hay datos históricos con el resultado real de cada paciente? ¿Cómo se medirá si el modelo es **mejor** que las reglas, y con qué tipo de error (no atender a un paciente grave es mucho peor que lo contrario)? ¿Podrá explicarse cada decisión a médicos y pacientes? Si las reglas funcionan y son exigidas por protocolo, quizá el ML deba **ayudar** (sugerir) y no **reemplazar**.

**B.** Porque los spammers **se adaptan**: si la lista bloquea "gratis", escriben "GR4TIS" o "gratuito". Mantener la lista a mano es una carrera perdida. Un modelo de ML aprende de **todas** las palabras del mensaje a la vez y se **reentrena** con los nuevos correos que los usuarios marcan (pregunta 5 de la lista: el patrón cambia). Lo vas a comprobar en la práctica siguiente.
</details>

### 8.3 🧩 Práctica integradora: filtro de spam, reglas contra ejemplos

Vas a construir dos filtros de spam, uno con **reglas** y otro con **ML**, y a compararlos con mensajes que ninguno ha visto. 5 pasos cortos; revisa cada respuesta antes de seguir.

**Paso 1 ▶️ — Los ejemplos.** 16 mensajes ya etiquetados para construir los filtros, y 8 mensajes **nuevos** para examinarlos.

In [ ]:
entrenamiento = pd.DataFrame([
    ("Ganaste un premio, reclama tu dinero gratis ahora", "spam"),
    ("Oferta exclusiva: gana dinero desde casa sin esfuerzo", "spam"),
    ("Tu cuenta será bloqueada, haz clic aquí para verificar", "spam"),
    ("Felicidades, eres el ganador de un iPhone gratis", "spam"),
    ("Préstamo inmediato sin buró, dinero en 5 minutos", "spam"),
    ("Haz clic aquí y gana un viaje gratis", "spam"),
    ("Últimas horas: premio en efectivo esperando por ti", "spam"),
    ("Verifica tus datos bancarios para recibir tu premio", "spam"),
    ("Hola, ¿nos vemos mañana para el café?", "ok"),
    ("Te envío el informe de ventas del mes", "ok"),
    ("La reunión se movió a las 4 de la tarde", "ok"),
    ("¿Me pasas la receta del pastel de ayer?", "ok"),
    ("Gracias por tu ayuda con el proyecto", "ok"),
    ("Recuerda que el viernes es el cumpleaños de mamá", "ok"),
    ("Adjunto la factura de este mes, cualquier duda me avisas", "ok"),
    ("¿Viste el partido de ayer? Qué golazo", "ok"),
], columns=["texto", "etiqueta"])

nuevos_msj = pd.DataFrame([
    ("Reclama ahora tu premio en efectivo", "spam"),
    ("Tus datos bancarios necesitan verificación urgente", "spam"),
    ("Dinero rápido desde casa, haz clic", "spam"),
    ("Eres el ganador de un viaje", "spam"),
    ("¿Nos vemos el viernes para el cumpleaños?", "ok"),
    ("Te paso el informe de la reunión", "ok"),
    ("Gracias por el pastel, estaba buenísimo", "ok"),
    ("Mañana te envío la factura del proyecto", "ok"),
], columns=["texto", "etiqueta"])

entrenamiento.head()

**Paso 2 ✏️ — El filtro de reglas.** Empieza con dos palabras sospechosas. Ejecuta y luego ✏️ añade las palabras que quieras a la lista para mejorarlo.

In [ ]:
PALABRAS_SOSPECHOSAS = ["gratis", "premio"]     # ✏️ añade palabras

def filtro_reglas(texto):
    return "spam" if any(p in texto.lower() for p in PALABRAS_SOSPECHOSAS) else "ok"

def acierto(filtro, tabla):
    return np.mean([filtro(t) == e for t, e in zip(tabla.texto, tabla.etiqueta)])

print(f"Reglas → acierto con los 16 de entrenamiento: {acierto(filtro_reglas, entrenamiento):.0%}")
print(f"Reglas → acierto con los 8 nuevos           : {acierto(filtro_reglas, nuevos_msj):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `["gratis", "premio"]`: **81 %** en entrenamiento y **62 %** en los nuevos (deja pasar "Tus datos bancarios…", "Dinero rápido…" y "Eres el ganador…").

Si añades palabras (p. ej. "dinero", "clic", "ganador", "datos bancarios"), mejorarás… pero fíjate en **cómo** lo haces: mirando los mensajes de prueba y escribiendo reglas a la medida. Eso es "hacer trampa" sin querer, y con el siguiente spam inventado volverás a fallar.
</details>

**Paso 3 ▶️ — El filtro de ML.** Este modelo (*Naive Bayes*, Lección 5.6) cuenta qué palabras aparecen en los mensajes de spam y en los normales, y **aprende solo** cuáles delatan al spam.

In [ ]:
vectorizador = CountVectorizer()                                   # convierte cada texto en conteos de palabras
X_entrenamiento = vectorizador.fit_transform(entrenamiento.texto)

filtro_ml = MultinomialNB().fit(X_entrenamiento, entrenamiento.etiqueta)   # ← aprende de los 16 ejemplos

print(f"ML → acierto con los 16 de entrenamiento: {filtro_ml.score(X_entrenamiento, entrenamiento.etiqueta):.0%}")
print(f"ML → acierto con los 8 nuevos           : {filtro_ml.score(vectorizador.transform(nuevos_msj.texto), nuevos_msj.etiqueta):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**100 % en los dos.** Nadie le dijo qué palabras eran sospechosas: aprendió que "dinero", "clic", "datos", "bancarios" o "ganador" aparecían en el spam, y que "informe", "viernes" o "pastel" aparecían en mensajes normales.

⚠️ No te emociones de más: con solo 8 mensajes de prueba, el 100 % puede ser en parte suerte. En la Lección 4.6 aprenderás a evaluar con más rigor.
</details>

**Paso 4 🔮 — El spammer contraataca.** Antes de ejecutar, predice qué dirá **cada filtro** para estos mensajes:
1. "Gana un iPhone GR4TIS"
2. "Ganaste el sorteo, reclama ya"
3. "Mamá, gana el equipo de ayer"

✍️ *Mis predicciones (reglas / ML):*

In [ ]:
trampas = ["Gana un iPhone GR4TIS", "Ganaste el sorteo, reclama ya", "Mamá, gana el equipo de ayer"]   # ✏️ inventa más

for msj in trampas:
    prob_spam = filtro_ml.predict_proba(vectorizador.transform([msj]))[0][1]
    print(f"{msj:<32} reglas: {filtro_reglas(msj):<5} · ML: {filtro_ml.predict(vectorizador.transform([msj]))[0]:<5} "
          f"(prob. de spam {prob_spam:.0%})")

<details><summary><b>👀 Ver respuesta</b></summary>

(Con la lista original de reglas, `["gratis", "premio"]`.)
1. **Reglas: ok ❌ · ML: spam ✅ (96 %).** "GR4TIS" engaña a la regla, pero el modelo se fija en "iPhone", que solo vio en spam.
2. **Reglas: ok ❌ · ML: spam ✅ (54 %).** Acierta, pero con poca seguridad: "sorteo" no lo vio nunca y "reclama" solo una vez.
3. **Reglas: ok ✅ · ML: ok ✅ (4 %).** Aunque contiene "gana", palabras como "mamá" y "ayer" pesan hacia mensaje normal.

El modelo combina **todas** las palabras a la vez, cosa que una lista de reglas no hace.
</details>

**Paso 5 ✋ — Conclusiones.**
1. ¿Qué filtro elegirías para una empresa? Da **dos** razones.
2. El filtro de ML acertó el mensaje 2 "por los pelos". ¿Qué harías para que mejore con el tiempo?
3. Repasa la lista de comprobación de la Sección 5: ¿este problema cumple las 5 preguntas?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El de **ML**: generaliza mejor a mensajes nuevos (combina todas las palabras) y se actualiza **reentrenándolo** con datos, sin que nadie reescriba reglas. (Un buen sistema real puede combinar ambos: reglas para casos obvios y ML para el resto.)
2. Añadir a los datos de entrenamiento los mensajes nuevos que los usuarios marcan como spam y **reentrenar** periódicamente. Así aprenderá palabras como "sorteo". Es parte del ciclo de vida de un modelo (Lección 1.5).
3. Sí: (1) hay patrón (el spam tiene un "estilo"); (2) las reglas a mano son frágiles; (3) hay datos etiquetados (los usuarios marcan spam); (4) un error de vez en cuando es tolerable; (5) el patrón cambia, porque los spammers se adaptan.
</details>

---
## 📚 Para profundizar
- 📘 **Curso gratuito:** *Machine Learning Crash Course* de Google, módulo introductorio (disponible en español).
- 📘 **Lectura:** Pedro Domingos, *The Master Algorithm*, cap. 1 (hay traducción: *El algoritmo maestro*).
- 🎮 **Interactivo:** *Teachable Machine* (Google): entrena un clasificador de imágenes con ejemplos de tu cámara y comprueba qué pasa cuando cambias el fondo o la luz.

---
⬅️ **Anterior:** [0.5 Proyecto del módulo: El catálogo de IA de la empresa 🗂️](../m00-bienvenida/0.5-proyecto-del-modulo.md) · ➡️ **Siguiente:** [1.2 Vocabulario esencial: dato, característica, etiqueta, modelo, predicción, parámetro e hiperparámetro](1.2-vocabulario-esencial.ipynb)